In [1]:
from langgraph.graph import StateGraph ,START ,END
from typing import TypedDict
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field


In [2]:
class cot_evaluation(BaseModel):
    cot_text: str
    cot_score: float
    
class doa_evaluation(BaseModel):
    doa_text: str
    doa_score: float

class lang_evaluation(BaseModel):
    
    lang_text: str
    lang_score: float


In [3]:
LLM=ChatOllama(model='llama3.2:latest')
cot_llm=LLM.with_structured_output(cot_evaluation)
doa_llm=LLM.with_structured_output(doa_evaluation)
lang_llm=LLM.with_structured_output(lang_evaluation)

In [4]:
class essay_evaluator(TypedDict):
    eassy: str
    cot_text: str
    doa_text: str
    lang_text: str
    cot_score: float
    doa_score: float
    lang_score: float
    summary: str
    avg_score: float


In [5]:
graph=StateGraph(essay_evaluator)

In [6]:
def cot_eval(state:essay_evaluator):
    propmt=f"rate this {state['eassy']} based on clarity of thought.provide text_feedback and score on scale of 0 to 10"
    llm_response=cot_llm.invoke(propmt)
    state['cot_text']=llm_response.cot_text
    state['cot_score']=llm_response.cot_score
    return {'cot_text':state['cot_text'],'cot_score':state['cot_score']}

def doa_eval(state:essay_evaluator):
    propmt=f"rate this {state['eassy']} based on depth of analysis.provide text_feedback and score on scale of 0 to 10"
    llm_response=doa_llm.invoke(propmt)
    state['doa_text']=llm_response.doa_text
    state['doa_score']=llm_response.doa_score
    return {'doa_text':state['doa_text'],'doa_score':state['doa_score']}

def lang_eval(state:essay_evaluator):
    propmt=f"rate this {state['eassy']} based on language of essay.provide text_feedback and score on scale of 0 to 10"
    llm_response=lang_llm.invoke(propmt)
    state['lang_text']=llm_response.lang_text
    state['lang_score']=llm_response.lang_score
    return {'lang_text':state['lang_text'],'lang_score':state['lang_score']}
  
def summary(state:essay_evaluator):
    propmt=f"""provide a summary of these feddbacks {state['lang_text'],state['doa_text'],state['cot_text']}"""
    summary=LLM.invoke(propmt)
    state['summary']=summary.content
    avg_score=(state['lang_score']+state['doa_score']+state['cot_score'])/3
    state['avg_score']=avg_score
    return state
    



In [7]:
graph.add_node('COT_EVALUATOR',cot_eval)
graph.add_node('DOA_EVALUATOR',doa_eval)
graph.add_node('language_EVALUATOR',lang_eval)
graph.add_node('summary',summary)

# define edge
graph.add_edge(START,'COT_EVALUATOR')
graph.add_edge(START,'DOA_EVALUATOR')
graph.add_edge(START,'language_EVALUATOR')
graph.add_edge('language_EVALUATOR','summary')
graph.add_edge('COT_EVALUATOR','summary')
graph.add_edge('DOA_EVALUATOR','summary')
graph.add_edge('summary',END)

# compile graph
llm_parallel_graph=graph.compile()






In [8]:
essay=LLM.invoke('provide an eassay on role of mugal on india')
initial_state={'eassy':essay}
reuslt=llm_parallel_graph.invoke(initial_state)

In [13]:
print(reuslt['avg_score'])

8.666666666666666
